# 使用 Milvus 和 DeepSeek 构建 RAG

DeepSeek 帮助开发者使用高性能语言模型构建和扩展 AI 应用。它提供高效的推理、灵活的 API 以及先进的专家混合 (MoE) 架构，用于强大的推理和检索任务。

在本教程中，我们将展示如何使用 Milvus 和 DeepSeek 构建一个检索增强生成 (RAG) 管道。

## 准备工作

### 依赖与环境

In [2]:
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0 ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   --------------------- ---------------- 524.3/914.9 kB 453.5 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 453.5 kB/s eta 0:00:01
   -------------------------------- ----- 786.4/914.9 kB 558.9 kB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 588.6 kB/s  0:00:01
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 622.7 kB/s eta 0:00:03
   --------- ------------------------

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\xiaozu121\\PycharmProjects\\deepseek-quickstart\\.venv\\share\\jupyter\\labextensions\\@jupyter-widgets\\jupyterlab-manager\\static\\vendors-node_modules_d3-color_src_color_js-node_modules_d3-format_src_defaultLocale_js-node_m-09b215.2643c43f22ad111f4f82.js.map'



---

In [2]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

### 准备数据

我们使用 Milvus 文档 2.4.x 中的 FAQ 页面作为我们 RAG 中的私有知识库，这是一个简单 RAG 管道的良好数据源。

下载 zip 文件并将文档解压到 `milvus_docs` 文件夹。

**建议在命令行执行下面命令**

数据已单独下载，解压在同包目录中

In [3]:
#!wget https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip
#!unzip -q milvus_docs_2.4.x_en.zip -d milvus_docs

我们从 `milvus_docs/en/faq` 文件夹加载所有 markdown 文件。对于每个文档，我们简单地使用 "# " 来分割文件中的内容，这样可以大致分离出 markdown 文件中每个主要部分的内容。

In [3]:
# 此段为Ubuntu下的写法
# from glob import glob
#
# text_lines = []
#
# for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
#     with open(file_path, "r") as file:
#         file_text = file.read()
#
#     text_lines += file_text.split("# ")


from pathlib import Path

text_lines = []

# 当前目录下所有 .md 文件
for file_path in Path(".").glob("milvus_docs/en/faq/*.md"):
    with open(file_path, "r", encoding="utf-8") as file:
        file_text = file.read()
    text_lines += file_text.split("# ")


In [4]:
len(text_lines)

72

### 准备 LLM 和 Embedding 模型

DeepSeek 支持 OpenAI 风格的 API，您可以使用相同的 API 进行微小调整来调用 LLM。

In [5]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

定义一个 embedding 模型，使用 `milvus_model` 来生成文本嵌入。我们以 `DefaultEmbeddingFunction` 模型为例，这是一个预训练的轻量级嵌入模型。

实际上因为没有OpenAI的API Key，而是使用了百炼的模型来处理

In [8]:
# from pymilvus import model as milvus_model
#
# embedding_model = milvus_model.DefaultEmbeddingFunction()

# 使用OpenAI连接模型 将文本数据处理成维度向量数据
# from pymilvus import model as milvus_model
#
# # OpenAI国内代理 https://api.apiyi.com/token
# embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
#     model_name='text-embedding-3-large', # Specify the model name
#     api_key='sk-XXX', # Provide your OpenAI API key
#     base_url='https://api.apiyi.com/v1',
#     dimensions=512
# )


# 使用百炼
import os
deepseek_api_key = os.getenv("ALI_API_KEY")

from pymilvus import model as milvus_model

embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-v4',
    api_key=deepseek_api_key,
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
    dimensions=512
)

生成一个测试嵌入并打印其维度和前几个元素。

上一步创建的512个维度

In [28]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])
# print(test_embedding)

512
[-0.02423961  0.03560013  0.03585002 -0.03654203 -0.01879002 -0.00445242
 -0.00890003  0.0340431  -0.08219554  0.18115313]


In [10]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[-0.02432832 -0.00198     0.04136878 -0.0215874  -0.01044843 -0.04005143
  0.01617993  0.08154769 -0.04381223  0.17329416]


## 将数据加载到 Milvus

### 创建 Collection

In [11]:
from pymilvus import MilvusClient

# milvus_client = MilvusClient(uri="./milvus_demo.db")
# 注意此处第一个参数为uri而非url!
milvus_client = MilvusClient(
    # 这个是在阿里云上，随时构建随时释放的 | 向量检索服务 Milvus 版
    # https://milvus.console.aliyun.com/?spm=5176.29313398.J_TC9GqcHi2edq9zUs9ZsDQ.1.6db25af8WP3Rn1#/region/cn-hangzhou/resource/all/detail/c-9d9e2d845603a56b/overview
    uri="http://c-9d9e2d845603a56b.milvus.aliyuncs.com:19530",
    token="root:G3fDtQak#6B5",
    db_name="default"
)

collection_name = "my_rag_collection"

In [ ]:
%%sql


关于 `MilvusClient` 的参数：

*   将 `uri` 设置为本地文件，例如 `./milvus.db`，是最方便的方法，因为它会自动利用 Milvus Lite 将所有数据存储在此文件中。
*   如果您有大规模数据，可以在 Docker 或 Kubernetes 上设置性能更高的 Milvus 服务器。在此设置中，请使用服务器 URI，例如 `http://localhost:19530`，作为您的 `uri`。
*   如果您想使用 Zilliz Cloud（Milvus 的完全托管云服务），请调整 `uri` 和 `token`，它们对应 Zilliz Cloud 中的 Public Endpoint 和 Api key。

检查 collection 是否已存在，如果存在则删除它。

In [12]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

创建一个具有指定参数的新 collection。

如果我们不指定任何字段信息，Milvus 将自动创建一个默认的 `id` 字段作为主键，以及一个 `vector` 字段来存储向量数据。一个保留的 JSON 字段用于存储非 schema 定义的字段及其值。

`metric_type` (距离度量类型):
     作用：定义如何计算向量之间的相似程度。
     例如：`IP` (内积) - 值越大通常越相似；`L2` (欧氏距离) - 值越小越相似；`COSINE` (余弦相似度) - 通常转换为距离，值越小越相似。
     选择依据：根据你的嵌入模型的特性和期望的相似性定义来选择。

 `consistency_level` (一致性级别):   更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
     作用：定义数据写入后，读取操作能多快看到这些新数据。
     例如：
         `Strong` (强一致性): 总是读到最新数据，可能稍慢。
         `Bounded` (有界过期): 可能读到几秒内旧数据，性能较好 (默认)。
         `Session` (会话一致性): 自己写入的自己能立刻读到。
         `Eventually` (最终一致性): 最终会读到新数据，但没时间保证，性能最好。
     选择依据：在数据实时性要求和系统性能之间做权衡。

简单来说：
 `metric_type`：怎么算相似。
 `consistency_level`：新数据多久能被读到。

In [13]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)
)

### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [14]:
# from tqdm import tqdm
#
# data = []
#
# doc_embeddings = embedding_model.encode_documents(text_lines)
#
# for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
#     data.append({"id": i, "vector": doc_embeddings[i], "text": line})
#
# milvus_client.insert(collection_name=collection_name, data=data)


# 上面是OpenAI模型时，可以一次插入2048条数据，但是百炼每次只能插入10条
# tqdm是一个进度条显示工具，框在for循环上，能看到进行到了哪一步
#   72条文本，步长10，分8段处理
from tqdm import tqdm

data = []
batch_size = 10

for i in tqdm(range(0, len(text_lines), batch_size), desc="处理文本批次"):
    batch_texts = text_lines[i:i+batch_size]
    # 分8次调用模型处理文本数据
    batch_embeddings = embedding_model.encode_documents(batch_texts)

    #   上面是每个文本1条1条的处理。这里是一段文本10条，在此基础上1条1条的处理，并且将输入的文本和输出的文本一一对应
    for j, (text, vector) in enumerate(zip(batch_texts, batch_embeddings)):
        data.append({
            "id": i + j,
            "vector": vector,
            "text": text
        })

# 一次性插入所有数据（Milvus 插入本身没有这个限制）
milvus_client.insert(collection_name=collection_name, data=data)

处理文本批次: 100%|██████████| 8/8 [00:02<00:00,  2.96it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

## 构建 RAG

### 检索查询数据

我们指定一个关于 Milvus 的常见问题。

In [15]:
question = "How is data stored in milvus?"

在 collection 中搜索该问题，并检索语义上最匹配的前3个结果。

把用户的问题（比如"Milvus如何存储数据？"）转换成向量，然后在数据库中找出与这个问题语义最相似的前3个文档片段。

encode_queries() 和之前用的 encode_documents() 的区别：

encode_documents()：给文档片段用（你插入的那些文本）

encode_queries()：给查询问题用（用户问的问题）

虽然底层调用的是同一个模型，但有些嵌入模型会对查询和文档做不同的优化处理。

In [16]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 使用内积距离（Inner Product）计算相似度。数值越大，表示两个向量越相似。
    output_fields=["text"],  # 返回 text 字段，这样你就能直接拿到匹配到的文档原文，而不只是向量的 ID
)

让我们看一下查询的搜索结果

In [19]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0] # 因为只查询了1个问题，所以结果在索引0的位置
]
print(json.dumps(retrieved_lines_with_distances, indent=4)) # 缩进4个空格，让输出更美观易读

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.7670297026634216
    ],
    [
        "How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float

### 使用 LLM 获取 RAG 响应

将检索到的文档转换为字符串格式。

In [20]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [21]:
context

' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float32, Float16, and BFloat16 vector types.\n\n- Binary vectors: Store

In [22]:
question

'How is data stored in milvus?'

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的文档组装而成的。

In [23]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

In [24]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。\n<context>\n Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data typ

使用 DeepSeek 提供的 `deepseek-chat` 模型根据提示生成响应。

In [26]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)

# 怎么理解 response.choices[0]： 第一个（也是唯一一个）生成的回答
print(response.choices[0].message.content)

Milvus stores data in two categories: inserted data (including vector data, scalar data, and collection-specific schema) is stored in persistent storage as incremental logs, supporting backends like MinIO, AWS S3, Google Cloud Storage, Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud COS. Metadata generated by Milvus modules is stored in etcd.

<translated>
Milvus 存储数据的方式分为两类：插入数据（包括向量数据、标量数据和集合特定模式）以增量日志形式存储在持久化存储中，支持 MinIO、AWS S3、Google Cloud Storage、Azure Blob Storage、阿里云 OSS 和腾讯云 COS 等后端；Milvus 模块生成的元数据存储在 etcd 中。
</translated>


怎么理解 response.choices[0]： 第一个（也是唯一一个）生成的回答？

问问AI后可知
# 你没有写 n 参数，所以使用默认值 n=1
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[...],
    # n=1 是默认值，意味着只生成1个回答
)

为什么需要多个回答？
有时候你可能想要模型的多个不同角度的回答：

创意任务：生成多个标题、多个广告语，从中挑选最好的

风险评估：看模型对同一个问题的不同表述是否一致

多样性：在聊天机器人中给用户提供多个回复选项

Mermaid代码，渲染流程图

flowchart LR
    A[用户问题] --> B[Milvus检索]
    B --> C[找到相关文档片段]
    C --> D[构建 USER_PROMPT<br>上下文 + 问题]
    E[SYSTEM_PROMPT<br>设定AI角色] --> F[DeepSeek API]
    D --> F
    F --> G[生成回答]
    G --> H[打印结果]